In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from IPython.display import display, Markdown, HTML

################
# CONFIGURATION
################

MODEL = "llama3.2:3b"

llm = ChatOllama(model = MODEL, temperature = 0.7)

# Quick connectivity test
test = llm.invoke("Say hello in one word")
print(f"\u2705 Connected to Ollama | MODEL : {MODEL}")
print(f"  Response : {test.content}")

✅ Connected to Ollama | MODEL : llama3.2:3b
  Response : Hello.


## 1. Component 1: Models — The AI Brain

### Experiment 1A: Message Types in Action

In [2]:
messages = [
    SystemMessage(content = "You are a AI tutor. Explain in one sentence."),
    HumanMessage(content = "What is API?")
]
response = llm.invoke(messages)
print(f"Response Message : {type(response).__name__}")
print(f"Response role : {response.type}")
print(f"Content : {response.content}")

Response Message : AIMessage
Response role : ai
Content : An Application Programming Interface (API) is a set of defined rules and protocols that enables different software systems to communicate with each other, allowing data to be shared, retrieved, or manipulated.


### Experiment 1B: The Runnable Protocol — `invoke()`, `stream()`, `batch()`

In [3]:
# --- invoke(): single input, full response ---
result = llm.invoke([
    SystemMessage(content = " You are a math tutor. Explain with step by step."),
    HumanMessage(content = "What is Area of Triangle explain with sum?")
])

print(result.content)

The area of a triangle! It's a fundamental concept in geometry that can be a bit tricky, but don't worry, I'm here to break it down for you.

**What is the Area of a Triangle?**

The area of a triangle is a measure of the amount of space inside the triangle. It's calculated using a specific formula that takes into account the length of the base and the height of the triangle.

**Step-by-Step Formula:**

To calculate the area of a triangle, you can use the following formula:

Area = (Base × Height) / 2

Let's break it down step by step:

1. **Find the Base:** The base is one side of the triangle. It's the horizontal line that connects two vertices (corners) of the triangle.
2. **Find the Height:** The height is the vertical distance from the base to the opposite vertex (corner) of the triangle. Think of it as a perpendicular line dropped from the top vertex to the base.
3. **Multiply Base and Height:** Multiply the length of the base by the height to get a product.
4. **Divide by 2:** F

In [4]:
# --- stream(): token by token ---
print("Stream : Token by Token")

for chunk in llm.stream("Name 3 LLM models. One line each"):
    print(chunk.content, end ="", flush = True)
print()

Stream : Token by Token
Here are three LLM (Large Language Model) models:

1. BERT (Bidirectional Encoder Representations from Transformers)
2. T5 (Text-to-Text Transfer Transformer)
3. XLNet (Extreme Language Modeling with Nested Embeddings and Headless Architecture)


In [5]:
# --- batch(): multiple inputs at once ---
print("batch() — Process multiple inputs\n")

queries = [
    "What is Rerrieval Augmented Generation? One sentencse",
    "What is model in AI? One sentence.",
    "What is lambda in python? One sentence."
]
result = llm.batch(queries)
for q, r in zip(queries, result):  # Here q in queries and r in results which runs into two loops
    print(f"Q : {q}")              # zip() : pairs them by position : queries 1 - answer 1, query 2 - answer - 2
    print(f"A : {r.content}\n")


batch() — Process multiple inputs

Q : What is Rerrieval Augmented Generation? One sentencse
A : I couldn't find any information on "Rerrieval Augmented Generation". It's possible that it's a new or emerging concept in the field of artificial intelligence, but without more context or information, I couldn't provide a definition. Can you please provide more details about what Rerrieval Augmented Generation is?

Q : What is model in AI? One sentence.
A : In Artificial Intelligence (AI), a model refers to a mathematical representation of a system, process, or relationship that is learned from data and used to make predictions, classify inputs, or generate outputs.

Q : What is lambda in python? One sentence.
A : In Python, a lambda function (also known as an anonymous function) is a small, single-line function definition that can be defined inline within a larger expression or statement, without having to declare it with the `def` keyword.



In [6]:
queries = [
    "What is Prompt. One sentence.",
    "What is output parser. One sentence.",
    "What is retrieval. One sentence"
]

result = llm.batch(queries)
for q,r in zip(queries, result):
    print(f"Q : {q}")
    print(f"A : {r.content}\n")

Q : What is Prompt. One sentence.
A : A prompt is a short text or phrase that provides input to an AI model, such as myself, to generate a response based on the context and information provided in the prompt.

Q : What is output parser. One sentence.
A : An Output Parser is a software tool or module that extracts, processes, and analyzes data from the output of a program or system, such as log files, CSV files, or database queries, to gain insights or perform specific tasks.

Q : What is retrieval. One sentence
A : Retrieval refers to the process of accessing and bringing back information, knowledge, or data that was previously learned or stored, often through recall or memory activation.



### Experiment 1C: Model Configuration Parameters

In [7]:
# Temperature comparison — same prompt, different randomness

prompt = "Give me a one sentence about LLM"

configs = [
    {"label": "Deterministic:", "temp": 0.0},
    {"label": "Balanced:", "temp": 0.5},
    {"label": "Creative:", "temp": 1.0}
]

for cfg in configs:
    model = ChatOllama(model = MODEL, temp = cfg["temp"])
    # Run twice to show consistency vs variety

    r1 = model.invoke(prompt).content
    r2 = model.invoke(prompt).content
    print(f"\n{cfg['label']}")
    print(f"Run 1 : {r1}")
    print(f"Run 2 : {r2}")
    print(f"Same? {'Yes' if r1 == r2 else 'No'}")


Deterministic:
Run 1 : A Large Language Model (LLM) is a type of artificial intelligence designed to process and generate human-like language, using complex algorithms and massive amounts of training data to learn patterns and relationships in language.
Run 2 : A Large Language Model (LLM) is a type of artificial intelligence designed to process and generate human-like language, often used in applications such as chatbots, language translation, and text summarization.
Same? No

Balanced:
Run 1 : A Large Language Model (LLM) is a type of artificial intelligence designed to process and generate human-like language, leveraging vast amounts of training data to learn patterns and relationships in natural language.
Run 2 : A Large Language Model (LLM) is a type of artificial intelligence that uses complex algorithms and vast amounts of training data to generate human-like language and respond to questions or tasks in a wide range of topics.
Same? No

Creative:
Run 1 : A Large Language Model

### Experiment 1D: Provider Swappability — The Power of Abstraction

In [8]:
# Demonstrate swappability with different Ollama configurations
# (Same pattern applies across OpenAI, Anthropic, Google, etc.)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. One sentence max"),
    ("human", "What is cricket?") 
])

# Two different "providers" (same model, different configs)
precise_llm = ChatOllama(model = MODEL, temperature = 0.0)
creative_llm = ChatOllama(model = MODEL, temperature = 1.0)

chain_a = prompt | precise_llm | parser 
chain_b = prompt | creative_llm | parser

print(f"Precise chain :")
print(chain_a.invoke({}))

print(f"\nCreative chain :")
print(chain_b.invoke({}))

Precise chain :
Cricket is a popular team sport played with a bat and ball, originating in England, where two teams of eleven players take turns batting and bowling to score runs and dismiss each other's batsmen.

Creative chain :
Cricket is a popular team sport played with a bat and ball on a rectangular field, requiring skill, strategy, and physical endurance to score runs and dismiss opponents' batsmen.


## 2. Component 2: Prompts — The Instructions

### Experiment 2A: PromptTemplate — Simple String Templates

In [9]:
# PromptTemplate — basic string with {variable} placeholders
prompt_a = PromptTemplate.from_template(
    "Explain {topic} to a {person} in {length}."
)

print(f"Template : {prompt_a.template}")
print(f"Variables : {prompt_a.input_variables}")

format = prompt_a.invoke({
    "topic" : "Prompt",
    "person" : "beginner",
    "length" : "2 sentences"
})

print(f"\nFormatted : {format.text}")

# Send to model
response = llm.invoke(format.text)
print(f"Response : {response.content}")

Template : Explain {topic} to a {person} in {length}.
Variables : ['length', 'person', 'topic']

Formatted : Explain Prompt to a beginner in 2 sentences.
Response : Prompt is a text-based input format used to communicate with artificial intelligence (AI) models, such as language generators or chatbots, to elicit specific responses or answers. By providing a clear and well-structured prompt, users can guide the AI model's output towards a desired outcome, such as generating creative writing, answering questions, or completing tasks.


### Experiment 2B: ChatPromptTemplate — The Modern Standard

In [10]:
# ChatPromptTemplate with role-based messages

chat_prompt = ChatPromptTemplate.from_messages([
    ("system"," You are a {role}. Always respond in {style}."),
    ("human", "{question}")
])
print(f"Variables : {chat_prompt.input_variables}")

format = chat_prompt.invoke({
    "role": "cricket coach",
    "style": "concise",
    "question": "Why batters difficult to hit the yorker ball?"
})
print(f"\n Formatted messages :")
for msg in format.messages:
    print(f" [{type(msg).__name__}], {msg.content}")

response = llm.invoke(format)
print(f"\nResponse: {response.content}")

Variables : ['question', 'role', 'style']

 Formatted messages :
 [SystemMessage],  You are a cricket coach. Always respond in concise.
 [HumanMessage], Why batters difficult to hit the yorker ball?

Response: Yorker's speed, bounce, and lack of width make it challenging. Batters struggle to react quickly enough due to:

1. Speed: Reaches batsman before they can react.
2. Bounce: Causes batsmen to lose their balance or footwork, making them off-balance.
3. Lack of width: Narrower margin for error, increasing the likelihood of a missed shot.

These factors combined make the yorker particularly difficult to hit consistently.


### Experiment 2C: MessagesPlaceholder — Dynamic Conversation History

In [11]:
# Template with a slot for conversation history
histroy_prompt =ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor.Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])
print(f"Variables : {histroy_prompt.input_variables}")

# Simulate conversation history
fake_history = [
    HumanMessage(content ="What is 5+2?"),
    AIMessage(content= "5 + 2 = 7"),
    HumanMessage(content= "Multiply by 5"),
    AIMessage(content= "7 x 5 = 35")
]

# Inject history + new question
messages = histroy_prompt.invoke({
    "chat_history": fake_history,
    "question" : "What was my 2nd question?"
})

print(f"\nFormatted messages :")
for msg in messages.messages:
    print(f"[{type(msg).__name__}] {msg.content}")

# The model can now "remember" the conversation
response = llm.invoke(messages)
print(f"\nResponse : {response.content}")

Variables : ['chat_history', 'question']

Formatted messages :
[SystemMessage] You are a helpful math tutor.Be concise.
[HumanMessage] What is 5+2?
[AIMessage] 5 + 2 = 7
[HumanMessage] Multiply by 5
[AIMessage] 7 x 5 = 35
[HumanMessage] What was my 2nd question?

Response : Your second question was "Multiply by 5".


In [19]:
# Full working conversation using MessagesPlaceholder

convo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Keep it one or two sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

print(f"Variables : {convo_prompt.input_variables}")
chain = convo_prompt | llm | StrOutputParser()

history = []
turns =[
    "Hi, I'm Kalai",
    "Give me 5 famous cricketer names. Only name?",
    "What is my name, what question i asked?"
]

for i, user_input in enumerate(turns,1):
    print("=" * 50)
    print(f"Turn {i}")
    print("=" * 50)
    print(f"User : {user_input}")

    response = chain.invoke({"history": history, "input": user_input})
    print(f"AI : {response}\n")

    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))


Variables : ['history', 'input']
Turn 1
User : Hi, I'm Kalai
AI : Nice to meet you, Kalai! Is there something I can help you with today?

Turn 2
User : Give me 5 famous cricketer names. Only name?
AI : 1. Sachin
2. MS Dhoni
3. Virat
4. AB de Villiers
5. Brian Lara

Turn 3
User : What is my name, what question i asked?
AI : Your name is Kalai and you asked me to give you 5 famous cricketer names.



In [13]:
# Define example input-output pairs
examples = [
    {"input":"strong", "output": "weak"},
    {"input":"fast", "output": "slow"},
    {"input":"up", "output": "down"}
]

# Template for formatting each example as a human-AI exchange
# Why : All examples have same format, instead of Human:happy, AI:sad, Human:fast, AI:slow, we create one template
examples_format = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Build the few-shot template. Here it takes i)list of eg ii)formatting template, and combines them
few_shot = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=examples_format
)

# Without FewShotChatMessagePromptTemplate :
'''ChatPromptTemplate.from_messages([
    ("system","..."),

    ("human","happy"),
    ("ai","sad"),

    ("human","tall"),
    ("ai","short"),

    ("human","fast"),
    ("ai","slow"),

    ("human","{input}") # If 50 examples, code becomes long
])'''

# Wrap in a full ChatPromptTemplate
final_prompt = ChatPromptTemplate.from_messages([
    ("system","You give the opposite word for the given word.One word only"),
    few_shot, #This inserts all the formatted examples

    ("human","{input}")

])
chain = final_prompt | llm | StrOutputParser()

input_words = ["high", "full", "old", "day"]

for word in input_words:
    result = chain.invoke({"input" : word})
    print(f"{word} -> {result}")
print(type(result))

high -> low
full -> empty
old -> new
day -> night
<class 'langchain_core.messages.base.TextAccessor'>


### Experiment 2F: Partial Prompts — Pre-Fill Variables

Pre-fill some variables at setup time, fill the rest at runtime.

In [14]:
from datetime import datetime

# Template with a mix of static and runtime variables
base_prompt = PromptTemplate.from_template(
    "Today is {date}. You are {role}. Answer : {question}"
)

partial_prompt = base_prompt.partial(
    date = datetime.now().strftime("%D-%M_%Y"),
    role= "MetaAI",
)

print(f"Variables : {base_prompt.input_variables}")
print(f"Partial variables : {partial_prompt.input_variables}")

formatted = partial_prompt.invoke({"question" : "What day is it?"})
print(f"Formatted : {formatted.text}")

response = llm.invoke(formatted.text)
print(f"Response : {response.content}")

Variables : ['date', 'question', 'role']
Partial variables : ['question']
Formatted : Today is 07/11/26-21_2026. You are MetaAI. Answer : What day is it?
Response : It's Tuesday, July 14th, 2026.


## 3. Component 3: Output Parsers — The Response Formatter

### Experiment 3B: CommaSeparatedListOutputParser — Get a Python List

### Experiment 3D: PydanticOutputParser — Production-Grade Schema Validation

The **gold standard** for production applications. Full type validation, default values, and clear error messages.

In [15]:
list_parser = CommaSeparatedListOutputParser()

print("Format Instructions:")
print(f"{list_parser.get_format_instructions()}")

# Build chain with format instructions injected into the prompt
list_prompt = ChatPromptTemplate.from_messages([
    ("system","You list items separated by commas. No numbering, no extra text. {format_instructions}"),
    ("human", "List 6 {question}.")
])

list_chain = list_prompt | llm | list_parser

result = list_chain.invoke({
    "question" : "Most used LLM in AI industry",
    "format_instructions" : list_parser.get_format_instructions()  
})

print(f"Type : {type(result).__name__}")
print(f"Response : {result}")



Format Instructions:
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
Type : list
Response : ['Google BERT', 'Meta T5', 'Hugging Face Transformers', 'IBM Deep Quest', 'OpenAI GPT-3', 'Amazon SageMaker']


### Experiment 3D: PydanticOutputParser — Production-Grade Schema Validation

The **gold standard** for production applications. Full type validation, default values, and clear error messages.

In [33]:
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    title : str = Field(description = "The movie title")
    genre : str= Field(description= "The movie genre")
    rating : int = Field(description= "Rating from 1-10")
    summary : str = Field(description= "One sentence summary")

pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

print("Format instructions (auto - generated) ")
print(pydantic_parser.get_format_instructions)

# Build chain with format instructions
pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie reviewer. Respond ONLY with valid JSON matching the schema."
     "No markdown, no extra text.\n{format_instructions}"),
    ("human", "Review the movie {movie}")
])

pydantic_chain = pydantic_prompt | llm | pydantic_parser

review = pydantic_chain.invoke({
    "format_instructions" : pydantic_parser.get_format_instructions(),
    "movie" : "Inception"  
})
print("=" * 50)
print(f"Type : {type(result).__name__}")
print(f"Title : {review.title}")
print(f"Genre : {review.genre}")
print(f"Rating : {review.rating}")
print(f"Summary : {review.summary}")
print("=" * 50)


print()
# We can get result using loop
# for item in review:
#     print(item)
for k, v in review.model_dump().items(): # model_dump() : review is a MovieReview object, not a dict 
    print(f"{k} : {v}")

# Loop over selected attributes
fields = ["title", "genre", "rating", "summary"]
print()
for field in fields:
    print(f"{field} : {getattr(review, field)}") # getattr(review, field) : same as result.title next iteration field = genre becomes result.genre 

Format instructions (auto - generated) 
<bound method PydanticOutputParser.get_format_instructions of PydanticOutputParser(pydantic_object=<class '__main__.MovieReview'>)>
Type : list
Title : Inception
Genre : Science Fiction, Action, Thriller
Rating : 8
Summary : A thief who specializes in entering people's dreams is tasked with planting an idea instead of stealing one, but things become complicated when the line between reality and dreams is blurred.

title : Inception
genre : Science Fiction, Action, Thriller
rating : 8
summary : A thief who specializes in entering people's dreams is tasked with planting an idea instead of stealing one, but things become complicated when the line between reality and dreams is blurred.

title : Inception
genre : Science Fiction, Action, Thriller
rating : 8
summary : A thief who specializes in entering people's dreams is tasked with planting an idea instead of stealing one, but things become complicated when the line between reality and dreams is blur

In [46]:
# Compare: Multiple movies through the same chain
movies = ["Ratatouille", "La La Land", "Baby Driver"]
for movie in movies:
    try:
        review = pydantic_chain.invoke({
            "movie" : movie,
            "format_instructions" : pydantic_parser.get_format_instructions()      
        })
        print(f"Title : {review.title} | Genre : {review.genre} | Rating : {review.rating} | Summary : {review.summary}")
    except Exception as e:
        print(f"Parse error movie {movie} : {e}")


Title : Ratatouille | Genre : Animated, Comedy, Drama | Rating : 7 | Summary : An ambitious rat named Remy teams up with a klutzy kitchen worker to create culinary masterpieces in Paris.
Title : La La Land | Genre : Romantic Musical | Rating : 8 | Summary : A young jazz pianist and an aspiring actress fall in love while chasing their dreams in Los Angeles.
Title : Baby Driver | Genre : Action, Thriller | Rating : 4 | Summary : A young getaway driver must escape his abusive boss and find love with a waitress while perfecting his driving skills.


---

## 5. Sandbox — Try It Yourself!

In [49]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Choose your parser
use_parser = "json"  # Options: "string", "list", "json"

if use_parser == "string":
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}")
    ])
    my_chain = my_prompt | llm | StrOutputParser()
    result = my_chain.invoke({"question": "What are the 3 core components in LangChain?"})

elif use_parser == "list":
    lp = CommaSeparatedListOutputParser()
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "List items separated by commas only. {fi}"),
        ("human", "List 4 {category}.")
    ])
    my_chain = my_prompt | llm | lp
    result = my_chain.invoke({"category": "output parser types", "fi": lp.get_format_instructions()})

elif use_parser == "json":
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "Respond with valid JSON only. No markdown."),
        ("human", 'Describe {topic}. JSON keys: "name", "purpose", "example_use".')
    ])
    my_chain = my_prompt | llm | JsonOutputParser()
    result = my_chain.invoke({"topic": "ChatPromptTemplate"})

print(f"Type:   {type(result).__name__}")
print(f"Result: {result}")

Type:   dict
Result: {'name': 'ChatPromptTemplate', 'purpose': 'A template for designing effective conversational prompts.', 'example_use': 'Use this template to create clear and concise questions or statements that guide conversations, improving the user experience.'}
